# Module Option E — Model Drift & Performance Monitoring

**Objectif** : surveiller la stabilité du modèle de pricing du Module 1 (LightGBM fréquence) dans le temps.
Détecte si les distributions des variables d'entrée ou des prédictions ont dérivé.

**Variables surveillées** : 10 variables les plus importantes selon SHAP (Bonus A).

**Split** : 80% référence / 20% récent

**Méthodes** :
- PSI (Population Stability Index) par variable
- Test KS sur les prédictions
- Dégradation Gini
- Score de stabilité global 0-100


In [1]:
# BLOC 0 — Dépendances
!pip install lightgbm scikit-learn --quiet


In [4]:
# =====================================================
# BLOC 1 — Chargement modèle + données
# =====================================================

import joblib
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
from scipy import stats as scipy_stats
from google.colab import drive

# -----------------------------------------------------
# Monter Google Drive
# -----------------------------------------------------
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/siap'

warnings.filterwarnings('ignore')
np.random.seed(42)

# -----------------------------------------------------
# Paramètres
# -----------------------------------------------------
SPLIT_REF  = 0.80
PSI_STABLE = 0.10
PSI_WATCH  = 0.25
KS_ALPHA   = 0.05

# -----------------------------------------------------
# Top 10 variables SHAP
# -----------------------------------------------------
TOP10_SHAP = [
    'BonusMalus',
    'VehAge',
    'DrivAge',
    'Density',
    'VehPower',
    'Region_R82',
    'Region_R91',
    'Region_R93',
    'BonusMalus_band_4.0',
    'VehAge_band_1.0'
]

# -----------------------------------------------------
# Chemins
# -----------------------------------------------------
MODEL_PATH = f'{BASE}/models/lgb_freq.pkl'
DATA_PATH  = f'{BASE}/data/freq_test.csv'
META_PATH  = f'{BASE}/models/deployment_meta.json'

# -----------------------------------------------------
# Chargement modèle
# -----------------------------------------------------
lgb_freq = joblib.load(MODEL_PATH)

# -----------------------------------------------------
# Chargement dataset
# -----------------------------------------------------
df_raw = pd.read_csv(DATA_PATH)

# -----------------------------------------------------
# Chargement métadonnées
# -----------------------------------------------------
with open(META_PATH, 'r') as f:
    meta = json.load(f)

feature_columns = meta['feature_columns']

# -----------------------------------------------------
# Vérifications
# -----------------------------------------------------
print(f'Modèle chargé     : LightGBM ({lgb_freq.num_trees()} arbres)')
print(f'Dataset           : {len(df_raw):,} lignes × {len(df_raw.columns)} colonnes')
print(f'Features modèle   : {len(feature_columns)}')

print("\nTop-10 SHAP disponibles dans freq_test.csv :")

for v in TOP10_SHAP:
    status = "OK" if v in df_raw.columns else "post-encoding"
    print(f'  {status}: {v}')

Mounted at /content/drive
Modèle chargé     : LightGBM (430 arbres)
Dataset           : 132,882 lignes × 59 colonnes
Features modèle   : 43

Top-10 SHAP disponibles dans freq_test.csv :
  OK: BonusMalus
  OK: VehAge
  OK: DrivAge
  OK: Density
  OK: VehPower
  OK: Region_R82
  OK: Region_R91
  OK: Region_R93
  post-encoding: BonusMalus_band_4.0
  post-encoding: VehAge_band_1.0


In [5]:
# BLOC 2 — Feature engineering + split reference/recent
def engineer_features(df, meta):
    df = df.copy()
    rare_regions = meta.get('rare_regions', [])
    rare_brands  = meta.get('rare_brands', [])

    df['DrivAge_band'] = pd.cut(
        df['DrivAge'], bins=[-1,24,34,49,64,200], labels=[1.0,2.0,3.0,4.0,5.0]
    ).astype(float)
    df['BonusMalus_band'] = pd.cut(
        df['BonusMalus'], bins=[-1,60,80,100,130,500], labels=[1.0,2.0,3.0,4.0,5.0]
    ).astype(float)
    df['VehAge_band'] = pd.cut(
        df['VehAge'], bins=[-1,1,5,10,15,100], labels=[1.0,2.0,3.0,4.0,5.0]
    ).astype(float)

    if 'Region' in df.columns:
        df['Region'] = df['Region'].apply(lambda r: r if r not in rare_regions else 'R_REF')
    if 'VehBrand' in df.columns:
        df['VehBrand'] = df['VehBrand'].apply(lambda b: b if b not in rare_brands else 'B1')

    cat_cols = ['Area','VehBrand','VehGas','Region','DrivAge_band','BonusMalus_band','VehAge_band']
    existing_cats = [c for c in cat_cols if c in df.columns]
    df_enc = pd.get_dummies(df, columns=existing_cats, drop_first=True, dtype=int)

    for col in feature_columns:
        if col not in df_enc.columns:
            df_enc[col] = 0
    return df_enc[feature_columns]

print('Feature engineering...')
X = engineer_features(df_raw, meta)
preds_all = lgb_freq.predict(X)

n_total = len(X)
n_ref   = int(n_total * SPLIT_REF)
X_ref    = X.iloc[:n_ref]
X_rec    = X.iloc[n_ref:]
pred_ref = preds_all[:n_ref]
pred_rec = preds_all[n_ref:]

print(f'Split 80/20 :')
print(f'  Reference : {len(X_ref):,} polices')
print(f'  Recent    : {len(X_rec):,} polices')
print(f'  Freq. moy. reference : {pred_ref.mean():.4f}')
print(f'  Freq. moy. recent    : {pred_rec.mean():.4f}')


Feature engineering...
Split 80/20 :
  Reference : 106,305 polices
  Recent    : 26,577 polices
  Freq. moy. reference : 0.1215
  Freq. moy. recent    : 0.1213


In [6]:
# BLOC 3 — PSI par variable
def compute_psi(ref_vals, rec_vals, n_bins=10):
    quantiles = np.linspace(0, 100, n_bins + 1)
    bins = np.unique(np.percentile(ref_vals, quantiles))
    if len(bins) < 3:
        bins = np.array([ref_vals.min()-0.01, ref_vals.mean(), ref_vals.max()+0.01])
    ref_counts, _ = np.histogram(ref_vals, bins=bins)
    rec_counts, _ = np.histogram(rec_vals, bins=bins)
    ref_pct = np.where(ref_counts == 0, 0.0001, ref_counts / len(ref_vals))
    rec_pct = np.where(rec_counts == 0, 0.0001, rec_counts / len(rec_vals))
    return float(np.sum((rec_pct - ref_pct) * np.log(rec_pct / ref_pct)))

def psi_status(v):
    if v < PSI_STABLE: return 'STABLE', 'OK'
    elif v < PSI_WATCH: return 'SURVEILLANCE', 'WARN'
    else: return 'DRIFT CRITIQUE', 'CRIT'

psi_results = []
for var in TOP10_SHAP:
    if var not in X_ref.columns:
        psi_results.append({'variable':var,'psi':None,'status':'NON DISPONIBLE',
                            'mean_ref':None,'mean_rec':None,'delta_mean_pct':None})
        continue
    rv = X_ref[var].values.astype(float)
    rr = X_rec[var].values.astype(float)
    pv = compute_psi(rv, rr)
    st, _ = psi_status(pv)
    psi_results.append({
        'variable': var, 'psi': round(pv,4), 'status': st,
        'mean_ref': round(float(rv.mean()),4),
        'mean_rec': round(float(rr.mean()),4),
        'delta_mean_pct': round((rr.mean()-rv.mean())/max(abs(rv.mean()),0.0001)*100,2)
    })

print('PSI par variable :')
print(f"{'Variable':<25} {'PSI':<10} {'Status':<20} {'Delta%'}")
print('-'*65)
for r in psi_results:
    if r['psi'] is None:
        print(f"  {r['variable']:<23} N/A        {r['status']}")
    else:
        print(f"  {r['variable']:<23} {r['psi']:<10.4f} {r['status']:<20} {r['delta_mean_pct']:+.2f}%")

n_stable = sum(1 for r in psi_results if r['status']=='STABLE')
n_watch  = sum(1 for r in psi_results if r['status']=='SURVEILLANCE')
n_drift  = sum(1 for r in psi_results if r['status']=='DRIFT CRITIQUE')
print(f'\nResume PSI : {n_stable} stables | {n_watch} surveillance | {n_drift} drift critique')


PSI par variable :
Variable                  PSI        Status               Delta%
-----------------------------------------------------------------
  BonusMalus              0.0001     STABLE               -0.12%
  VehAge                  0.0005     STABLE               +0.31%
  DrivAge                 0.0004     STABLE               +0.26%
  Density                 0.0001     STABLE               -0.21%
  VehPower                0.0001     STABLE               +0.22%
  Region_R82              0.0000     STABLE               -1.14%
  Region_R91              0.0000     STABLE               -2.20%
  Region_R93              0.0000     STABLE               +0.21%
  BonusMalus_band_4.0     0.0000     STABLE               +1.32%
  VehAge_band_1.0         0.0000     STABLE               +0.00%

Resume PSI : 10 stables | 0 surveillance | 0 drift critique


In [7]:
# BLOC 4 — Drift des predictions (KS test + PSI predictions)
from scipy.stats import ks_2samp

ks_stat, ks_pvalue = ks_2samp(pred_ref, pred_rec)
pred_drift_detected = bool(ks_pvalue < KS_ALPHA)

psi_pred = compute_psi(pred_ref, pred_rec, n_bins=20)
pred_psi_status, _ = psi_status(psi_pred)

pred_stats = {
    'mean_ref': float(pred_ref.mean()),
    'mean_rec': float(pred_rec.mean()),
    'std_ref':  float(pred_ref.std()),
    'std_rec':  float(pred_rec.std()),
    'p5_ref':   float(np.percentile(pred_ref,5)),
    'p95_ref':  float(np.percentile(pred_ref,95)),
    'p5_rec':   float(np.percentile(pred_rec,5)),
    'p95_rec':  float(np.percentile(pred_rec,95)),
    'delta_mean_pct': float((pred_rec.mean()-pred_ref.mean())/pred_ref.mean()*100),
}

print(f'Freq. moy. reference : {pred_stats["mean_ref"]:.5f}')
print(f'Freq. moy. recent    : {pred_stats["mean_rec"]:.5f}')
print(f'Delta moyenne        : {pred_stats["delta_mean_pct"]:+.2f}%')
print(f'\nTest KS :')
print(f'  Statistique KS : {ks_stat:.4f}')
print(f'  p-value        : {ks_pvalue:.6f}')
print(f'  Drift detecte  : {"OUI" if pred_drift_detected else "NON"} (seuil a={KS_ALPHA})')
print(f'\nPSI predictions : {psi_pred:.4f} — {pred_psi_status}')


Freq. moy. reference : 0.12147
Freq. moy. recent    : 0.12126
Delta moyenne        : -0.18%

Test KS :
  Statistique KS : 0.0068
  p-value        : 0.285094
  Drift detecte  : NON (seuil a=0.05)

PSI predictions : 0.0011 — STABLE


In [8]:
# BLOC 5 — Degradation Gini
from sklearn.metrics import roc_auc_score

gini_ref = gini_rec = gini_delta = None
gini_degradation = False

if 'ClaimNb' in df_raw.columns:
    y_all = (df_raw['ClaimNb'] > 0).astype(int).values
    y_ref = y_all[:n_ref]
    y_rec = y_all[n_ref:]
    try:
        auc_ref = roc_auc_score(y_ref, pred_ref)
        auc_rec = roc_auc_score(y_rec, pred_rec)
        gini_ref = 2*auc_ref - 1
        gini_rec = 2*auc_rec - 1
        gini_delta = gini_rec - gini_ref
        gini_degradation = gini_delta < -0.02
        print(f'Gini reference : {gini_ref:.4f} ({gini_ref*100:.2f}%)')
        print(f'Gini recent    : {gini_rec:.4f} ({gini_rec*100:.2f}%)')
        print(f'Delta Gini     : {gini_delta:+.4f} ({gini_delta*100:+.2f} points)')
        print(f'Degradation    : {"OUI" if gini_degradation else "NON"} (seuil: -2 pts)')
    except Exception as e:
        print(f'Gini non calculable : {e}')
else:
    print('ClaimNb absent — Gini non calculable')
    print('Stabilite evaluee via PSI + KS uniquement')


Gini reference : 0.2505 (25.05%)
Gini recent    : 0.2664 (26.64%)
Delta Gini     : +0.0159 (+1.59 points)
Degradation    : NON (seuil: -2 pts)


In [9]:
# BLOC 6 — Score de stabilite global (0-100)
valid_psi = [r for r in psi_results if r['psi'] is not None]
if valid_psi:
    psi_scores = [1.0 if r['psi']<PSI_STABLE else 0.5 if r['psi']<PSI_WATCH else 0.0
                  for r in valid_psi]
    score_psi = np.mean(psi_scores) * 40
else:
    score_psi = 20.0

score_pred = 30.0 if psi_pred<PSI_STABLE else 15.0 if psi_pred<PSI_WATCH else 0.0
score_ks   = 0.0 if pred_drift_detected else 15.0

if gini_delta is None:
    score_gini = 7.5
elif not gini_degradation:
    score_gini = 15.0
else:
    score_gini = max(0, 15.0 + gini_delta*100)

stability_score = round(score_psi + score_pred + score_ks + score_gini, 1)

if stability_score >= 80:
    stability_class = 'STABLE'
    stability_label = 'Modele stable — aucune action requise'
    stability_color = 'VERT'
elif stability_score >= 55:
    stability_class = 'SURVEILLANCE'
    stability_label = 'Surveillance recommandee — reevaluer dans 30 jours'
    stability_color = 'AMBRE'
else:
    stability_class = 'DRIFT'
    stability_label = 'Drift detecte — reentranement recommande'
    stability_color = 'ROUGE'

drift_alerts = []
for r in psi_results:
    if r['psi'] is not None and r['psi'] >= PSI_STABLE:
        drift_alerts.append({
            'type':'PSI_VARIABLE','variable':r['variable'],'psi':r['psi'],
            'status':r['status'],
            'message':f"{r['variable']} : PSI={r['psi']:.4f} ({r['status']}), delta={r['delta_mean_pct']:+.2f}%"
        })
if pred_drift_detected:
    drift_alerts.append({
        'type':'KS_PREDICTIONS','variable':'Predictions',
        'message':f'Drift predictions (KS={ks_stat:.4f}, p={ks_pvalue:.6f})'
    })
if gini_degradation:
    drift_alerts.append({
        'type':'GINI_DEGRADATION','variable':'Gini',
        'message':f'Degradation Gini : {gini_delta*100:+.2f} pts ({gini_ref*100:.2f}% -> {gini_rec*100:.2f}%)'
    })

print('='*60)
print('SCORE DE STABILITE — MODEL DRIFT MONITORING')
print('='*60)
print(f'Score          : {stability_score} / 100')
print(f'Classification : {stability_class} ({stability_color})')
print(f'Verdict        : {stability_label}')
print(f'\nDetail du score :')
print(f'  PSI variables   : {score_psi:.1f} / 40')
print(f'  PSI predictions : {score_pred:.1f} / 30')
print(f'  KS test         : {score_ks:.1f} / 15')
print(f'  Gini            : {score_gini:.1f} / 15')
print(f'\nAlertes ({len(drift_alerts)}) :')
for a in drift_alerts:
    print(f'  WARN {a["message"]}')
if not drift_alerts:
    print('  OK Aucune alerte — modele stable')


SCORE DE STABILITE — MODEL DRIFT MONITORING
Score          : 100.0 / 100
Classification : STABLE (VERT)
Verdict        : Modele stable — aucune action requise

Detail du score :
  PSI variables   : 40.0 / 40
  PSI predictions : 30.0 / 30
  KS test         : 15.0 / 15
  Gini            : 15.0 / 15

Alertes (0) :
  OK Aucune alerte — modele stable


In [10]:
# BLOC 7 — Checklist de validation
print('CHECKLIST DE VALIDATION — MODULE OPTION E')
print('-'*55)
checks = [
    ('Modele LightGBM charge', lgb_freq is not None),
    ('Dataset > 10 000 polices', len(df_raw) > 10_000),
    ('Split 80/20 correct', abs(len(X_ref)/len(X) - 0.80) < 0.01),
    ('PSI calcule sur >= 5 variables', sum(1 for r in psi_results if r['psi'] is not None) >= 5),
    ('PSI predictions calcule', psi_pred is not None),
    ('Test KS calcule', ks_stat is not None),
    ('Score 0-100', 0 <= stability_score <= 100),
    ('Classification valide', stability_class in ['STABLE','SURVEILLANCE','DRIFT']),
    ('Predictions coherentes (moy > 0.05)', pred_ref.mean() > 0.05),
    ('Export pret', True),
]
all_ok = True
for lbl, ok in checks:
    print(f"  {'OK    ' if ok else 'ERREUR'} {lbl}")
    if not ok: all_ok = False
print()
if all_ok:
    print('TOUTES LES VALIDATIONS PASSEES')
    print(f'  Score stabilite : {stability_score}/100 — {stability_class}')
    print(f'  PSI predictions : {psi_pred:.4f} — {pred_psi_status}')
    print(f'  KS p-value      : {ks_pvalue:.6f}')
else:
    print('Certaines validations ont echoue')


CHECKLIST DE VALIDATION — MODULE OPTION E
-------------------------------------------------------
  OK     Modele LightGBM charge
  OK     Dataset > 10 000 polices
  OK     Split 80/20 correct
  OK     PSI calcule sur >= 5 variables
  OK     PSI predictions calcule
  OK     Test KS calcule
  OK     Score 0-100
  OK     Classification valide
  OK     Predictions coherentes (moy > 0.05)
  OK     Export pret

TOUTES LES VALIDATIONS PASSEES
  Score stabilite : 100.0/100 — STABLE
  PSI predictions : 0.0011 — STABLE
  KS p-value      : 0.285094


In [11]:
# BLOC 8 — Export artifacts
import os, shutil
from datetime import datetime
os.makedirs('drift_artifacts', exist_ok=True)

report = {
    'generated_at'    : datetime.now().isoformat(),
    'model_monitored' : 'Module 1 — LightGBM Frequency Model',
    'dataset'         : f'freq_test.csv ({len(df_raw):,} polices)',
    'methodology': {
        'split_ref_pct'         : SPLIT_REF*100,
        'n_ref'                 : int(n_ref),
        'n_recent'              : int(len(X_rec)),
        'psi_stable_threshold'  : PSI_STABLE,
        'psi_watch_threshold'   : PSI_WATCH,
        'ks_alpha'              : KS_ALPHA,
        'variables_monitored'   : TOP10_SHAP,
    },
    'stability_score' : stability_score,
    'stability_class' : stability_class,
    'stability_color' : stability_color,
    'verdict'         : stability_label,
    'alerts'          : drift_alerts,
    'score_detail': {
        'psi_variables'   : round(score_psi,1),
        'psi_predictions' : round(score_pred,1),
        'ks_test'         : round(score_ks,1),
        'gini'            : round(score_gini,1),
    },
    'psi_results': psi_results,
    'prediction_drift': {
        'psi'            : round(psi_pred,4),
        'psi_status'     : pred_psi_status,
        'ks_statistic'   : round(float(ks_stat),4),
        'ks_pvalue'      : round(float(ks_pvalue),6),
        'drift_detected' : bool(pred_drift_detected),
        'mean_ref'       : round(float(pred_ref.mean()),5),
        'mean_rec'       : round(float(pred_rec.mean()),5),
        'delta_mean_pct' : round(pred_stats['delta_mean_pct'],2),
    },
    'performance': {
        'gini_ref'    : round(gini_ref,4) if gini_ref else None,
        'gini_recent' : round(gini_rec,4) if gini_rec else None,
        'gini_delta'  : round(gini_delta,4) if gini_delta else None,
        'degradation' : bool(gini_degradation),
    },
}

with open('drift_artifacts/drift_report.json','w') as f:
    json.dump(report, f, indent=2, default=str)

print('Artifacts exportes dans drift_artifacts/ :')
for p in Path('drift_artifacts').iterdir():
    print(f'  {p.name} ({p.stat().st_size//1024+1} KB)')

shutil.make_archive('drift_artifacts','zip','drift_artifacts')
try:
    from google.colab import files
    files.download('drift_artifacts.zip')
    print('OK drift_artifacts.zip telecharge')
except ImportError:
    print('OK drift_artifacts.zip pret (hors Colab)')


Artifacts exportes dans drift_artifacts/ :
  drift_report.json (3 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

OK drift_artifacts.zip telecharge
